# S1.2 · Concentration and association

*Phase S, statistics and probability · S1, part 2 of 4 · about 2 hours*

S1.1 described one variable at a time. This lesson asks two new questions. How unequally is something shared out: who gets the citations? And how do two variables move together: do longer reference lists attract more citations? Both questions have traps that are typical of bibliographic data. A concentration measure depends on who is counted, one outlier can manufacture a correlation, and a comparison between groups can reverse when you split the data.

**Run this lesson:** [in your browser (JupyterLite)](https://sbrzt.github.io/personal-course/lite/lab/index.html?path=lessons/phase-s/s1-2-concentration-association.ipynb) · [in Google Colab](https://colab.research.google.com/github/sbrzt/personal-course/blob/main/lessons/phase-s/s1-2-concentration-association.ipynb)

**By the end you can**

- measure concentration with top shares, the Lorenz curve and the Gini coefficient;
- choose between Pearson's and Spearman's correlation, and show how one outlier can decide Pearson's;
- recognise Simpson's paradox, and compute a field-normalised citation indicator.

The exercises work as in S1.1: replace each `...`, run the cell, read the check.

In [ ]:
# ⚙️ Setup: run this cell first
import sys, pathlib, subprocess

if sys.platform == "emscripten":            # in the browser (JupyterLite)
    import piplite
    await piplite.install(["personal-course"])
elif "google.colab" in sys.modules:         # in Google Colab
    repo = "git+https://github.com/sbrzt/personal-course"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"personal-course @ {repo}"], check=True)
else:                                       # on your computer
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:    # find the course folder
        if (folder / "course" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course import data, plots, stats
from course.checks import s1_2 as check

plots.use_style()
print("Ready.")

## 1 · Who gets the citations?

We reuse the two simulated datasets of S1.1: the citations received by 5,000 papers, and the references given by 300 articles.

In [ ]:
cites = data.citation_counts()
refs = data.reference_counts()

The simplest concentration measure is a **top share**: the share of all citations that go to the most cited 10% (or 1%) of papers.

### ✏️ Exercise 1 · Top shares

Compute `top10_share`, the share of all citations received by the 10% most cited papers (the top 500 of 5,000), and `top1_share`, the share received by the top 1% (50 papers).

In [ ]:
ranked = np.sort(cites)[::-1]          # most cited first
top10_share = ...
top1_share = ...

check.ex1(top10_share, top1_share)

<details>
<summary>💡 Hint</summary>

`ranked[:500].sum()` is the number of citations received by the 500 most cited papers.

</details>

<details>
<summary>✅ Solution</summary>

```python
ranked = np.sort(cites)[::-1]
top10_share = ranked[:500].sum() / ranked.sum()
top1_share = ranked[:50].sum() / ranked.sum()

check.ex1(top10_share, top1_share)
```

Selecting the papers at or above the 90th percentile (`cites >= np.percentile(cites, 90)`) gives a slightly different answer: here it selects 507 papers, not 500, because several papers share the count at the threshold. Ties decide how many papers you get. Slicing the sorted array takes exactly 500.

</details>

### The Lorenz curve and the Gini coefficient

A top share looks at one point. The **Lorenz curve** shows them all. Sort the papers from least to most cited; for each share of papers on the horizontal axis, plot the share of all citations that those papers hold. If every paper had the same number of citations, the curve would be the diagonal. The more it sags below the diagonal, the more unequal the distribution.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
plots.lorenz(cites, ax=ax, label="citations received (5,000 papers)",
             color=plots.BLUE)
plots.lorenz(refs, ax=ax, label="references given (300 articles)",
             color=plots.ORANGE, equality=False)
ax.set_title("Citations are far more concentrated than references")
ax.legend(loc="upper left")
plt.show()

The **Gini coefficient** turns the curve into one number: the area between the diagonal and the curve, divided by the whole area under the diagonal. It is 0 when everyone holds the same amount and approaches 1 when one unit holds everything. For values sorted from smallest to largest, $x_{(1)} \le \dots \le x_{(n)}$:

$$G = \frac{2 \sum_{i=1}^{n} i \, x_{(i)}}{n \sum_{i=1}^{n} x_{(i)}} - \frac{n + 1}{n}$$

### ✏️ Exercise 2 · Your own Gini coefficient

Implement `my_gini(x)` with the formula above.

In [ ]:
def my_gini(x):
    x = np.sort(np.asarray(x, dtype=float))    # smallest first
    n = x.size
    i = np.arange(1, n + 1)                    # ranks 1, 2, ..., n
    return ...

check.ex2(my_gini)

<details>
<summary>💡 Hint</summary>

`np.sum(i * x)` is the numerator's sum, and `x.sum()` the denominator's.

</details>

<details>
<summary>✅ Solution</summary>

```python
def my_gini(x):
    x = np.sort(np.asarray(x, dtype=float))
    n = x.size
    i = np.arange(1, n + 1)
    return 2 * np.sum(i * x) / (n * x.sum()) - (n + 1) / n

check.ex2(my_gini)
```

An equivalent definition: the Gini coefficient is the mean absolute difference between all pairs of units, divided by twice the mean. Some software multiplies the result by n / (n − 1), a "sample" correction; with thousands of units the difference does not matter.

</details>

In [ ]:
print(f"Gini, references given:   {stats.gini(refs):.2f}")
print(f"Gini, citations received: {stats.gini(cites):.2f}")
print(f"Gini, cited papers only:  {stats.gini(cites[cites > 0]):.2f}")

> **🔬 Metascience: who is in the denominator?** The last line drops the uncited papers, and the Gini coefficient falls from 0.72 to 0.66. A concentration measure describes a population, so it changes when the population does. Reference lists show you only the authors who were cited at least once: a Gini coefficient computed from them leaves out everyone who was never cited. Always say who was counted. You will meet this in S1.4.

> **🤖 Why this matters for AI.** Recommender systems and AI search tools are judged partly by how concentrated their suggestions are, and the Gini coefficient over recommended items is a standard measure of "popularity bias". Algaba et al. (2025) asked GPT-4, GPT-4o and Claude 3.5 to suggest references for recent machine-learning papers. The suggestions were biased towards highly cited work even more than the human-written reference lists were, which could reinforce the concentration you just measured. S1.3 shows how such a feedback loop works.

## 2 · Do longer reference lists attract more citations?

`data.refs_and_cites()` holds 400 simulated articles and one long review article, with the number of references each gives and the number of citations each receives.

In [ ]:
rc = data.refs_and_cites()
rc.tail(3)

In [ ]:
fig, ax = plt.subplots()
art = rc[rc["type"] == "article"]
ax.scatter(art["references"], art["citations"] + 1, s=14,
           color=plots.BLUE, label="articles")
review = rc[rc["type"] == "review"]
ax.scatter(review["references"], review["citations"] + 1, s=60, marker="D",
           color=plots.ORANGE, label="one review article")
ax.set(xscale="log", yscale="log", xlabel="references given",
       ylabel="citations received + 1",
       title="A weak relationship, and one far-away point")
ax.legend()
plt.show()

**Pearson's correlation coefficient** *r* measures *linear* association. It is built from each point's distance from the means, so extreme points weigh the most. **Spearman's rank correlation** ρ (rho) is Pearson's *r* computed on the ranks of the values instead of the values themselves. It measures whether one variable tends to rise with the other, in any monotonic way, and a single extreme point can shift it only by a little.

### ✏️ Exercise 3 · Pearson or Spearman?

Compute both coefficients between `references` and `citations`: for all 401 rows (`pearson_all`, `spearman_all`), and for the articles alone, without the review (`pearson_articles`, `spearman_articles`).

In [ ]:
articles = rc[rc["type"] == "article"]
pearson_all = ...
spearman_all = ...
pearson_articles = ...
spearman_articles = ...

check.ex3(pearson_all, spearman_all, pearson_articles, spearman_articles)

<details>
<summary>💡 Hint</summary>

`rc["references"].corr(rc["citations"])` gives Pearson's *r*; add `method="spearman"` for Spearman's ρ.

</details>

<details>
<summary>✅ Solution</summary>

```python
articles = rc[rc["type"] == "article"]
pearson_all = rc["references"].corr(rc["citations"])
spearman_all = rc["references"].corr(rc["citations"], method="spearman")
pearson_articles = articles["references"].corr(articles["citations"])
spearman_articles = articles["references"].corr(articles["citations"],
                                                method="spearman")

check.ex3(pearson_all, spearman_all, pearson_articles, spearman_articles)
```

One review among 401 rows lifts Pearson's *r* from 0.22 to 0.83: from "weak" to "strong". Spearman's ρ stays at 0.22. With heavy-tailed data, report Spearman's ρ (or Kendall's τ, another rank correlation), or compute Pearson's *r* on log-transformed values, and always look at the scatter plot first.

</details>

> **🤖 Why this matters for AI.** When researchers test whether an automatic score agrees with human judgements (for example a metric for machine translation or summarisation), they usually report rank correlations such as Spearman's ρ or Kendall's τ, for the reason you just saw: a few extreme systems or texts should not decide the verdict.

## 3 · Simpson's paradox: an open-access puzzle

`data.oa_articles()` holds 2,400 simulated articles from two fields with very different citation habits. Each article is either open access (OA) or closed. The data are constructed to make a point, so they are not evidence about real open access; real studies are discussed at the end of the section.

In [ ]:
oa = data.oa_articles()
oa.sample(5, random_state=1)

### ✏️ Exercise 4 · Simpson's paradox

Compute:

1. `overall`, the mean citations of OA and of closed articles: a Series indexed by `oa` (False, True);
2. `by_field`, the mean citations by field and access type: a DataFrame with one row per field and the columns False and True.

In [ ]:
overall = ...
by_field = ...

check.ex4(overall, by_field)

<details>
<summary>💡 Hint</summary>

`oa.groupby("oa")["citations"].mean()` gives `overall`. Group by two columns, `["field", "oa"]`, and then call `.unstack()` to turn the second level into columns.

</details>

<details>
<summary>✅ Solution</summary>

```python
overall = oa.groupby("oa")["citations"].mean()
by_field = oa.groupby(["field", "oa"])["citations"].mean().unstack()

check.ex4(overall, by_field)
```

</details>

In [ ]:
print("Share of articles that are OA, by field:")
print(oa.groupby("field")["oa"].mean().round(2))

Overall, OA articles receive about twice as many citations as closed ones. Within each field, they receive *fewer*. Both statements are true. The overall comparison mixes the fields: three quarters of the life-science articles are OA against a quarter of the humanities articles, and life-science articles are cited far more whatever their access. Field affects both access and citations, so it **confounds** the comparison. When an association in the pooled data reverses within every subgroup, it is called **Simpson's paradox**.

Which comparison is right depends on the question, not on the data alone. To ask whether making an article OA increases its citations, comparing within fields is closer to fair, but still not enough: authors may choose OA for their best papers, for example. S5 introduces causal diagrams (DAGs) to reason about such choices explicitly.

### Field normalisation

Bibliometrics handles differences between fields by **normalising**: each paper's citations are divided by the mean citations of papers in the same field (and usually the same year and document type). A normalised score of 1 means "cited as much as the field average". The mean of these scores over a group of papers is the **mean normalised citation score** (MNCS), one of the indicators of the CWTS Leiden Ranking.

### ✏️ Exercise 5 · Field-normalised citation scores

Add a column `ncs` with each article's citations divided by the mean citations of its field. Then compute `mncs`, the mean of `ncs` for OA and for closed articles: a Series indexed by `oa`.

In [ ]:
oa["ncs"] = ...
mncs = ...

check.ex5(mncs)

<details>
<summary>💡 Hint</summary>

`oa.groupby("field")["citations"].transform("mean")` returns, for every row, the mean of that row's field: an array as long as `oa`, ready for division.

</details>

<details>
<summary>✅ Solution</summary>

```python
oa["ncs"] = oa["citations"] / oa.groupby("field")["citations"].transform("mean")
mncs = oa.groupby("oa")["ncs"].mean()

check.ex5(mncs)
```

The raw ratio says that OA articles get 1.94 times the citations of closed ones; the field-normalised ratio says 0.90. Normalising by field removed the effect of the field mix. Note that the MNCS is a mean again, with the mean's sensitivity to extreme values. That is why the Leiden Ranking also reports percentile-based indicators, such as the share of an institution's papers that belong to the 10% most cited of their field.

</details>

> **🔬 Metascience: the open access citation advantage.** Whether OA articles are cited more is one of the most studied questions in bibliometrics. A systematic review of 134 studies (Langham-Putrow, Bakker & Riegelman, 2021) found that 64 reported an advantage, 37 found none, and 32 found one only in some subsets of their data. Studies differ, among other things, in how they handle confounders such as field, article age and self-selection. You will come back to this question with regression and causal diagrams in S5.

## 4 · Wrap-up

### ✍️ Explain it

A colleague writes: "Across our 2,400 articles, open-access ones are cited twice as much: OA doubles impact." In at most 120 words, explain what is wrong with this inference and what you would compute instead.

*Your answer:*

<details>
<summary>A model answer</summary>

The comparison pools two fields whose citation levels differ eightfold, and OA is three times as common in the highly cited field. The raw difference therefore mostly reflects which field the OA articles come from: field is a confounder. Within each field, OA articles are in fact cited slightly less. I would compare OA and closed articles within fields, for instance with field-normalised scores (MNCS) or percentile indicators, and report the within-field differences. Even then, the comparison shows an association, not an effect: authors choose OA for some papers and not others, and the reasons may also affect citations.

</details>

### Check your progress

In [ ]:
check.summary();

### Self-check

1. Two journals have the same mean citations per article. One has a Gini coefficient of 0.3, the other of 0.8. Describe each journal in one sentence.
2. Spearman's ρ is Pearson's *r* computed on what?
3. Think of a Simpson's paradox that could arise in your own field, for example in library loans or digitisation projects.

<details>
<summary>Answers</summary>

1. In the first, most articles receive roughly the journal's typical number of citations. In the second, a few articles receive most of the citations and most articles receive few.
2. On the ranks of the two variables.
3. One possibility: digitised items are borrowed (or viewed) more than non-digitised ones overall, but less within each collection, because the collections chosen for digitisation were the popular ones to begin with.

</details>

### Done when

- [ ] Exercises 1 to 5 show ✅ in `check.summary()`.
- [ ] You can explain Simpson's paradox with the open-access example, without notes.
- [ ] You have written your 120-word explanation.

### Further reading

- Bickel, P. J., Hammel, E. A., & O'Connell, J. W. (1975). Sex bias in graduate admissions: Data from Berkeley. *Science*, 187(4175), 398–404. <https://doi.org/10.1126/science.187.4175.398>
- Pearl, J., Glymour, M., & Jewell, N. P. (2016). *Causal Inference in Statistics: A Primer*, chapter 1. Wiley.
- Waltman, L. (2016). A review of the literature on citation impact indicators. *Journal of Informetrics*, 10(2), 365–391. <https://doi.org/10.1016/j.joi.2016.02.007>
- Langham-Putrow, A., Bakker, C., & Riegelman, A. (2021). Is the open access citation advantage real? A systematic review of the citation of open access and subscription-based articles. *PLoS ONE*, 16(6), e0253129. <https://doi.org/10.1371/journal.pone.0253129>
- Algaba, A., Mazijn, C., Holst, V., Tori, F., Wenmackers, S., & Ginis, V. (2025). Large language models reflect human citation patterns with a heightened citation bias. In *Findings of the Association for Computational Linguistics: NAACL 2025* (pp. 6844–6879). <https://doi.org/10.18653/v1/2025.findings-naacl.381>

**Next:** S1.3 · Heavy tails and the laws of bibliometrics.